### 설치

In [ ]:
%%capture
import os, re

# Unsloth installs are environment-sensitive. Keep dependencies fresh enough for
# Qwen3 MoE while avoiding a hard pin that can break on new CUDA images.
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -U unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets>=4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install -U "transformers>=4.56.0" "trl>=0.22.0" "datasets>=4.3.0" accelerate bitsandbytes hf_transfer

### Unsloth

목표: `hitoshura25/crossvul` 데이터셋을 사용해 `unsloth/Qwen3-Coder-30B-A3B-Instruct`를 DefAPI 보안 수정 응답에 맞게 파인튜닝합니다.

이 노트북의 태스크는 수학 추론이 아닙니다. DefAPI는 취약 코드를 스캔하고, 모델이 취약점 설명, 안전한 수정 코드, 수정 이유, 추가 주의사항을 반환하길 기대합니다.

Qwen의 기본 채팅 템플릿은 유지하고, system/user/assistant 메시지만 서비스 계약에 맞게 구성합니다.

In [ ]:
import gc
import math
import os
import torch

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

# Optional MoE backend override. grouped_mm needs torch 2.9+; if it fails, leave
# this unset and let Unsloth choose the backend for the current runtime.
# os.environ.setdefault("UNSLOTH_MOE_BACKEND", "grouped_mm")

if not torch.cuda.is_available():
    raise RuntimeError("Qwen3-Coder-30B-A3B fine-tuning requires a CUDA GPU runtime.")

print(torch.cuda.get_device_name(0))
print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")

In [ ]:
model_name = os.getenv("BASE_MODEL", "unsloth/Qwen3-Coder-30B-A3B-Instruct")
dataset_name = os.getenv("DATASET_NAME", "hitoshura25/crossvul")

# Use RUN_MODE=production for the full run. Smoke mode is intentionally small so
# the pipeline can be validated before spending GPU hours.
run_mode = os.getenv("RUN_MODE", "smoke").lower()
seed = int(os.getenv("SEED", "3407"))
max_seq_length = int(os.getenv("MAX_SEQ_LENGTH", "8192"))
lora_rank = int(os.getenv("LORA_RANK", "32"))

output_dir = os.getenv("OUTPUT_DIR", "outputs/qwen3-coder-crossvul-lora")
hub_model_id = os.getenv("HUB_MODEL_ID", "").strip() or None
push_to_hub = os.getenv("PUSH_TO_HUB", "false").lower() == "true"
if push_to_hub and hub_model_id is None:
    raise ValueError("Set HUB_MODEL_ID when PUSH_TO_HUB=true.")

if run_mode == "production":
    max_train_samples = int(os.getenv("MAX_TRAIN_SAMPLES", "0"))  # 0 means all
    max_eval_samples = int(os.getenv("MAX_EVAL_SAMPLES", "256"))
    max_steps = int(os.getenv("MAX_STEPS", "-1"))
    num_train_epochs = float(os.getenv("NUM_TRAIN_EPOCHS", "1"))
    learning_rate = float(os.getenv("LEARNING_RATE", "2e-5"))
    eval_steps = int(os.getenv("EVAL_STEPS", "100"))
    save_steps = int(os.getenv("SAVE_STEPS", "100"))
else:
    max_train_samples = int(os.getenv("MAX_TRAIN_SAMPLES", "256"))
    max_eval_samples = int(os.getenv("MAX_EVAL_SAMPLES", "64"))
    max_steps = int(os.getenv("MAX_STEPS", "50"))
    num_train_epochs = float(os.getenv("NUM_TRAIN_EPOCHS", "1"))
    learning_rate = float(os.getenv("LEARNING_RATE", "2e-4"))
    eval_steps = int(os.getenv("EVAL_STEPS", "25"))
    save_steps = int(os.getenv("SAVE_STEPS", "50"))

print({
    "run_mode": run_mode,
    "model_name": model_name,
    "dataset_name": dataset_name,
    "max_seq_length": max_seq_length,
    "lora_rank": lora_rank,
    "max_train_samples": max_train_samples,
    "max_eval_samples": max_eval_samples,
    "max_steps": max_steps,
    "learning_rate": learning_rate,
    "push_to_hub": push_to_hub,
    "hub_model_id": hub_model_id,
})

In [ ]:
from unsloth import FastLanguageModel

load_in_4bit = os.getenv("LOAD_IN_4BIT", "false").lower() == "true"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
    fast_inference = False,  # vLLM fast path is not the right path for MoE LoRA training.
    trust_remote_code = True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "gate_up_proj",
    ],
    lora_alpha = lora_rank * 2,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = seed,
    bias = "none",
)

model.print_trainable_parameters()

### DefAPI 서비스 채팅 형식

Qwen3-Coder Instruct에서는 tokenizer의 기본 채팅 템플릿인 `<|im_start|>role ... <|im_end|>` 형식을 유지합니다. GRPO/수학 예제용 커스텀 템플릿으로 덮어쓰면 모델이 이미 학습한 role boundary가 깨질 수 있습니다.

이번 SFT에서 중요한 것은 메시지 계약입니다.

- system: 보안 코딩 assistant의 역할과 행동
- user: 언어, CWE 메타데이터, 취약 코드
- assistant: DefAPI가 요구하는 네 개 섹션과 수정 코드 block

In [ ]:
SYSTEM_MESSAGE = (
    "You are a secure coding assistant for DefAPI. "
    "Analyze vulnerable code and provide safe, practical fixes."
)

RESPONSE_FORMAT = """응답은 반드시 다음 네 섹션을 순서대로 포함해야 한다.
1. 취약점 설명
2. 안전한 수정 코드
3. 수정 이유
4. 추가 주의사항"""

USER_INSTRUCTION = (
    "작업:\n"
    "다음 취약 코드를 분석하고 안전한 코드로 수정하라.\n\n"
    f"출력 형식:\n{RESPONSE_FORMAT}"
)

print(SYSTEM_MESSAGE)
print(RESPONSE_FORMAT)

아래 미리보기는 Qwen의 기본 Instruct 형식처럼 보여야 합니다. 생성은 `<|im_start|>assistant` 뒤에서 시작하고, SFT에서 실제 손실 타깃은 assistant 답변만 남기는 것이 이상적입니다.

In [ ]:
preview_messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": f"{USER_INSTRUCTION}\n\n언어: python\nCWE: CWE-89\nCWE 설명: SQL Injection\n\n취약 코드:\n```python\nquery = \"SELECT * FROM users WHERE name = '\" + username + \"'\"\n```"},
]

print(tokenizer.apply_chat_template(preview_messages, tokenize=False, add_generation_prompt=True))

단일 학습 예시는 실제 DefAPI 클라이언트가 사용하는 네 섹션 응답 형식과 같은 assistant 응답을 포함해야 합니다.

In [ ]:
example_answer = """1. 취약점 설명
사용자 입력을 SQL 문자열에 직접 연결해 SQL Injection이 발생할 수 있다.

2. 안전한 수정 코드
```python
def find_user(conn, username):
    return conn.execute("SELECT * FROM users WHERE name = ?", (username,)).fetchall()
```

3. 수정 이유
파라미터 바인딩을 사용하면 입력값이 SQL 문법이 아니라 데이터로 처리된다.

4. 추가 주의사항
사용 중인 DB 드라이버의 placeholder 문법을 확인하고, 권한이 제한된 DB 계정을 사용한다."""

print(tokenizer.apply_chat_template(
    preview_messages + [{"role": "assistant", "content": example_answer}],
    tokenize=False,
))

### CrossVul SFT 데이터 준비

`hitoshura25/crossvul`은 Hugging Face Dataset 카드 기준 9,313개의 before/after 보안 코드 쌍을 제공합니다. 주요 컬럼은 다음과 같습니다.

- `cwe_id`
- `cwe_description`
- `language`
- `vulnerable_code`
- `fixed_code`
- `file_pair_id`
- `source`
- `language_dir`

완성도 있는 파인튜닝을 위해 이 노트북은 전체 데이터를 pandas로 복사하지 않고 HF Dataset에서 바로 `filter/map`합니다. 또한 assistant 응답 구간만 loss로 학습되도록 `prompt_text`와 `text`를 함께 보관합니다.

In [ ]:
from datasets import load_dataset

REQUIRED_COLUMNS = ["cwe_id", "cwe_description", "language", "vulnerable_code", "fixed_code"]
OPTIONAL_COLUMNS = ["file_pair_id", "source", "language_dir"]

raw_dataset = load_dataset(dataset_name, split="train")
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(raw_dataset.column_names))
if missing_columns:
    raise ValueError(f"Missing expected CrossVul columns: {missing_columns}")


def clean_text(value):
    if value is None:
        return ""
    return str(value).strip()


def has_required_content(row):
    return all(clean_text(row.get(column)) for column in REQUIRED_COLUMNS)


dataset = raw_dataset.filter(has_required_content, desc="Drop rows with missing CrossVul fields")
dataset = dataset.shuffle(seed=seed)

if max_train_samples > 0 and run_mode != "production":
    # In smoke mode, sample before tokenization to keep iteration fast.
    dataset = dataset.select(range(min(len(dataset), max_train_samples + max_eval_samples)))

dataset

각 CrossVul row를 DefAPI 채팅 SFT 샘플로 변환합니다.

In [ ]:
FENCE_LANGUAGE_ALIASES = {
    "c++": "cpp",
    "c#": "csharp",
    "js": "javascript",
    "py": "python",
    "rb": "ruby",
}


def normalize_language(value):
    language = clean_text(value).lower() or "text"
    return FENCE_LANGUAGE_ALIASES.get(language, language.replace(" ", "-"))


def fence_code(code, language):
    fence = "````" if "```" in code else "```"
    return f"{fence}{language}\n{code}\n{fence}"


def build_user_prompt(row):
    language = normalize_language(row.get("language_dir") or row.get("language"))
    cwe_id = clean_text(row["cwe_id"]) or "unknown"
    cwe_description = clean_text(row["cwe_description"]) or "No description provided"
    vulnerable_code = clean_text(row["vulnerable_code"])

    return (
        f"{USER_INSTRUCTION}\n\n"
        f"언어: {language}\n"
        f"CWE: {cwe_id}\n"
        f"CWE 설명: {cwe_description}\n\n"
        f"취약 코드:\n{fence_code(vulnerable_code, language)}"
    )


def build_assistant_answer(row):
    language = normalize_language(row.get("language_dir") or row.get("language"))
    cwe_id = clean_text(row["cwe_id"]) or "해당 CWE"
    cwe_description = clean_text(row["cwe_description"]) or "보안 취약점"
    fixed_code = clean_text(row["fixed_code"])

    return (
        "1. 취약점 설명\n"
        f"이 코드는 {cwe_id}({cwe_description}) 유형의 취약점과 관련될 수 있다. "
        "취약 코드에서 신뢰할 수 없는 입력, 외부 데이터, 권한 경계, 메모리 경계, 출력 인코딩, "
        "또는 보안 검증이 누락된 흐름을 확인해야 한다.\n\n"
        "2. 안전한 수정 코드\n"
        f"{fence_code(fixed_code, language)}\n\n"
        "3. 수정 이유\n"
        "수정 코드는 취약한 처리 흐름을 안전한 API, 검증, 인코딩, 권한 제한, 경계 검사, "
        "또는 제한된 실행 방식으로 대체해 공격 가능성을 낮춘다.\n\n"
        "4. 추가 주의사항\n"
        "실제 서비스에 적용하기 전 의존성 버전, 프레임워크 권장 방식, 권한 범위, 회귀 테스트, "
        "그리고 SAST 재검사 결과를 함께 확인하라."
    )


def format_dataset(row):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": build_user_prompt(row)},
        {"role": "assistant", "content": build_assistant_answer(row)},
    ]
    prompt_text = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {
        "messages": messages,
        "prompt_text": prompt_text,
        "text": text,
        "cwe_id": clean_text(row["cwe_id"]),
        "language": normalize_language(row.get("language_dir") or row.get("language")),
        "file_pair_id": clean_text(row.get("file_pair_id")),
    }


keep_columns = [column for column in REQUIRED_COLUMNS + OPTIONAL_COLUMNS if column in dataset.column_names]
formatted_dataset = dataset.map(
    format_dataset,
    remove_columns=[column for column in dataset.column_names if column not in keep_columns],
    desc="Build DefAPI chat prompts",
)
formatted_dataset

변환 결과가 의도대로 만들어졌는지 확인합니다.

In [ ]:
print(formatted_dataset[0]["text"][:4000])

`max_seq_length`를 초과하는 예시는 제외합니다. CrossVul에는 500KB에 가까운 긴 파일도 있으므로, 완성도 있는 학습에서는 `MAX_SEQ_LENGTH=8192` 이상을 기본으로 두고 너무 긴 샘플은 별도 repo-level retrieval 학습 데이터로 분리하는 편이 좋습니다.

In [ ]:
def count_tokens(row):
    return {"N": len(tokenizer(row["text"], add_special_tokens=False)["input_ids"])}


formatted_dataset = formatted_dataset.map(count_tokens, desc="Count tokens")
formatted_dataset = formatted_dataset.filter(lambda row: row["N"] <= max_seq_length, desc="Keep samples within context")

if run_mode == "production" and max_train_samples > 0:
    formatted_dataset = formatted_dataset.select(range(min(len(formatted_dataset), max_train_samples + max_eval_samples)))

lengths = formatted_dataset["N"]
print({
    "usable_rows": len(formatted_dataset),
    "min_tokens": min(lengths) if lengths else None,
    "avg_tokens": round(sum(lengths) / len(lengths), 2) if lengths else None,
    "max_tokens": max(lengths) if lengths else None,
})

if len(formatted_dataset) < 10:
    raise ValueError("Too few samples remain after length filtering. Increase MAX_SEQ_LENGTH or inspect CrossVul rows.")

assistant 응답 구간만 loss에 들어가도록 직접 tokenization과 label masking을 수행합니다. 이렇게 하면 모델이 system/user 프롬프트를 그대로 외우는 비중을 줄이고, 실제 보안 수정 응답 품질에 학습 용량을 더 씁니다.

In [ ]:
split_dataset = formatted_dataset.train_test_split(test_size=0.1, seed=seed)
train_formatted = split_dataset["train"]
eval_formatted = split_dataset["test"]

if max_train_samples > 0:
    train_formatted = train_formatted.select(range(min(len(train_formatted), max_train_samples)))
if max_eval_samples > 0:
    eval_formatted = eval_formatted.select(range(min(len(eval_formatted), max_eval_samples)))


def tokenize_and_mask(row):
    full = tokenizer(row["text"], add_special_tokens=False, truncation=False)
    prompt = tokenizer(row["prompt_text"], add_special_tokens=False, truncation=False)
    labels = list(full["input_ids"])
    prompt_len = min(len(prompt["input_ids"]), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    return {
        "input_ids": full["input_ids"],
        "attention_mask": full["attention_mask"],
        "labels": labels,
    }


train_dataset = train_formatted.map(
    tokenize_and_mask,
    remove_columns=train_formatted.column_names,
    desc="Tokenize train split",
)
eval_dataset = eval_formatted.map(
    tokenize_and_mask,
    remove_columns=eval_formatted.column_names,
    desc="Tokenize eval split",
)

print(train_dataset)
print(eval_dataset)

In [ ]:
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16

training_args = SFTConfig(
    output_dir = output_dir,
    per_device_train_batch_size = int(os.getenv("PER_DEVICE_TRAIN_BATCH_SIZE", "1")),
    per_device_eval_batch_size = int(os.getenv("PER_DEVICE_EVAL_BATCH_SIZE", "1")),
    gradient_accumulation_steps = int(os.getenv("GRADIENT_ACCUMULATION_STEPS", "4")),
    warmup_ratio = float(os.getenv("WARMUP_RATIO", "0.03")),
    max_steps = max_steps,
    num_train_epochs = num_train_epochs,
    learning_rate = learning_rate,
    logging_steps = int(os.getenv("LOGGING_STEPS", "5")),
    eval_strategy = "steps",
    eval_steps = eval_steps,
    save_strategy = "steps",
    save_steps = save_steps,
    save_total_limit = int(os.getenv("SAVE_TOTAL_LIMIT", "2")),
    optim = os.getenv("OPTIM", "adamw_8bit"),
    weight_decay = float(os.getenv("WEIGHT_DECAY", "0.01")),
    lr_scheduler_type = os.getenv("LR_SCHEDULER_TYPE", "cosine"),
    seed = seed,
    bf16 = bf16,
    fp16 = fp16,
    max_grad_norm = float(os.getenv("MAX_GRAD_NORM", "0.3")),
    gradient_checkpointing = True,
    remove_unused_columns = False,
    dataset_kwargs = {"skip_prepare_dataset": True},
    report_to = os.getenv("REPORT_TO", "none"),
    push_to_hub = push_to_hub,
    hub_model_id = hub_model_id,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer = tokenizer,
    model = model,
    padding = True,
    pad_to_multiple_of = 8,
    label_pad_token_id = -100,
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    data_collator = data_collator,
    args = training_args,
)

trainer.args

In [ ]:
train_result = trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

metrics = train_result.metrics
metrics["train_samples"] = len(train_dataset)
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

if push_to_hub:
    trainer.push_to_hub()

metrics

모델이 보류 샘플 형태의 프롬프트에서도 DefAPI 응답 형식을 따르는지 확인합니다.

In [ ]:
FastLanguageModel.for_inference(model)

messages = eval_formatted[0]["messages"][:2]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

from transformers import TextStreamer
inputs = tokenizer(text, return_tensors="pt").to(model.device)
_ = model.generate(
    **inputs,
    temperature=0.1,
    top_p=0.8,
    max_new_tokens=1024,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
    use_cache=True,
)

이 스모크 체크가 끝나면 저장이나 다음 실행 전에 데이터셋을 GPU/CPU 메모리에서 정리합니다.

In [ ]:
del raw_dataset, dataset, formatted_dataset, train_formatted, eval_formatted, train_dataset, eval_dataset
torch.cuda.empty_cache()
gc.collect()

<a name="Save"></a>
### VLLM용 float16 저장

`float16`으로 바로 저장할 수 있습니다. float16 저장은 `merged_16bit`, int4 저장은 `merged_4bit`를 선택하세요. 대안으로 `lora` adapter만 저장할 수도 있습니다. Hugging Face 계정에 업로드하려면 `push_to_hub_merged`를 사용합니다. 개인 token은 https://huggingface.co/settings/tokens 에서 발급할 수 있습니다. 더 많은 배포 옵션은 [문서](https://unsloth.ai/docs/basics/inference-and-deployment)를 참고하세요.

In [ ]:
# Local adapter save is already done by trainer.save_model(output_dir).
# Use environment variables instead of hardcoding tokens in the notebook.
hf_token = os.getenv("HF_TOKEN")
adapter_repo = os.getenv("ADAPTER_REPO", hub_model_id or "HF_USERNAME/qwen3-coder-crossvul-lora")
merged_repo = os.getenv("MERGED_REPO", "HF_USERNAME/qwen3-coder-crossvul-merged")

# Just LoRA adapters
if False:
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
if False:
    model.push_to_hub(adapter_repo, token=hf_token)
    tokenizer.push_to_hub(adapter_repo, token=hf_token)

# Merge to 16bit
if False:
    model.save_pretrained_merged("qwen3_coder_crossvul_16bit", tokenizer, save_method="merged_16bit")
if False:
    model.push_to_hub_merged(merged_repo, tokenizer, save_method="merged_16bit", token=hf_token)

# Merge to 4bit
if False:
    model.save_pretrained_merged("qwen3_coder_crossvul_4bit", tokenizer, save_method="merged_4bit")
if False:
    model.push_to_hub_merged(merged_repo, tokenizer, save_method="merged_4bit", token=hf_token)

### GGUF / llama.cpp 변환

`GGUF` / `llama.cpp` 형식 저장도 지원합니다. 내부적으로 `llama.cpp`를 clone하고 기본 quantization은 `q8_0`을 사용합니다. `q4_k_m` 같은 다른 방식도 사용할 수 있습니다. 로컬 저장은 `save_pretrained_gguf`, Hugging Face 업로드는 `push_to_hub_gguf`를 사용하세요.

지원되는 quantization 방식 예시입니다. 전체 목록은 [문서](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)를 참고하세요.
* `q8_0` - 빠르게 변환할 수 있습니다. 리소스 사용량은 높지만 일반적으로 무난합니다.
* `q4_k_m` - 권장 옵션입니다. attention.wv와 feed_forward.w2 tensor의 절반에는 Q6_K를, 나머지에는 Q4_K를 사용합니다.
* `q5_k_m` - 권장 옵션입니다. attention.wv와 feed_forward.w2 tensor의 절반에는 Q6_K를, 나머지에는 Q5_K를 사용합니다.

[**새 기능**] 파인튜닝 후 Ollama로 자동 export하려면 [Ollama 노트북](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)을 참고하세요.

In [ ]:
hf_token = os.getenv("HF_TOKEN")
gguf_repo = os.getenv("GGUF_REPO", "HF_USERNAME/qwen3-coder-crossvul-gguf")

# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("qwen3_coder_crossvul_gguf", tokenizer)
if False:
    model.push_to_hub_gguf(gguf_repo, tokenizer, token=hf_token)

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen3_coder_crossvul_gguf", tokenizer, quantization_method="f16")
if False:
    model.push_to_hub_gguf(gguf_repo, tokenizer, quantization_method="f16", token=hf_token)

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen3_coder_crossvul_gguf", tokenizer, quantization_method="q4_k_m")
if False:
    model.push_to_hub_gguf(gguf_repo, tokenizer, quantization_method="q4_k_m", token=hf_token)

# Save multiple GGUF options in one upload.
if False:
    model.push_to_hub_gguf(
        gguf_repo,
        tokenizer,
        quantization_method=["q4_k_m", "q8_0", "q5_k_m"],
        token=hf_token,
    )